# T4 — Gradient flow versus Hamiltonian flow

**Facts used** (classical; `scripts/verify/kuramoto_einstein_refutation.py`
check 1, claim `einstein-from-kuramoto-chain-a`, refuted).

1. With symmetric coupling, $\theta' = -\partial V/\partial\theta$ with
   $V = -K\sum\cos(\theta_j - \theta_i)$: $V$ is a Lyapunov function and the
   linearization about a locked state is self-adjoint (Strogatz 2000).
2. The signature computed here: a small perturbation of a locked chain
   spreads diffusively, width $\sim t^{1/2}$; the same chain with inertia
   (a wave equation) spreads ballistically, width $\sim t^{1}$. The verify
   script reports 0.5 versus 0.989 at $N=401$, 4000 steps; this notebook
   runs a smaller, faster version of the same computation.
3. Curriculum T4 names curvature as the holonomy of clock transport; that
   material is cited only (Sagnac 1913; MTW sec. 17). Nothing here derives it.

In [ ]:
import sys, math, json, cmath, random
from fractions import Fraction
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CATALOG.md").exists())
sys.path.insert(0, str(_root / "notebooks"))
from nbkit import ROOT, show_svg, catalog, verify, mutant_must_fail, falsify
import termplot
print("repo root:", ROOT.name)

In [ ]:
def spread(inertial, N=201, K=1.0, dt=0.02, steps=2000, every=400):
    th = [0.0] * N
    w = [0.0] * N
    th[N // 2] = 1e-3
    out = []
    V0 = None
    for k in range(1, steps + 1):
        F = [K * (math.sin(th[(i + 1) % N] - th[i]) + math.sin(th[i - 1] - th[i])) for i in range(N)]
        if inertial:
            for i in range(N):
                w[i] += F[i] * dt
                th[i] += w[i] * dt
        else:
            for i in range(N):
                th[i] += F[i] * dt
        if k % every == 0:
            m0 = sum(abs(v) for v in th)
            m2 = sum(abs(v) * (i - N // 2) ** 2 for i, v in enumerate(th))
            V = -K * sum(math.cos(th[(i + 1) % N] - th[i]) for i in range(N))
            out.append((k * dt, math.sqrt(m2 / m0), V))
    return out

res = {name: spread(flag) for name, flag in (("overdamped", False), ("inertial", True))}
expo = {}
for name, rows in res.items():
    (t0, w0, _), (t1, w1, _) = rows[0], rows[-1]
    expo[name] = math.log(w1 / w0) / math.log(t1 / t0)
    print(f"{name:>10}: widths {[round(w, 2) for _, w, _ in rows]}  exponent {expo[name]:.3f}")
print(termplot.plot_xy([(math.log(t), math.log(w)) for t, w, _ in res['overdamped']] +
                       [(math.log(t), math.log(w)) for t, w, _ in res['inertial']], width=50, height=12,
                       title="ln width vs ln t (lower: overdamped ~1/2, upper: inertial ~1)", xlabel="ln t", ylabel="ln w"))

In [ ]:
def check(model="overdamped", target=0.5):
    return abs(expo[model] - target) < 0.1

falsify(check, {"inertial": lambda: {"model": "inertial"}})

## The Lyapunov function decreases; the inertial energy does not

In [ ]:
Vs = [V for _, _, V in res["overdamped"]]
print("overdamped V(t) samples:", [f"{v:.9f}" for v in Vs])
print("monotone non-increasing:", all(b <= a + 1e-12 for a, b in zip(Vs, Vs[1:])))

## Falsifier: the verify script's `inertial` mutant must fail

In [ ]:
rc, out = verify("kuramoto_einstein_refutation.py", mutant="inertial")
mutant_must_fail("inertial", rc, out)